In [17]:
import json

with open("constitutions.json", "r", encoding="utf-8") as file:
    const = json.load(file)

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
import re
import nltk

In [19]:
nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/arocha/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Error loading corpora/stopwords: Package
[nltk_data]     'corpora/stopwords' not found in index
[nltk_data] Error loading help/tagsets: Package 'help/tagsets' not
[nltk_data]     found in index


In [20]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")

[nltk_data] Downloading package punkt to /Users/arocha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/arocha/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/arocha/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [21]:
import json
import spacy
import pandas as pd

# Load your JSON
with open("constitutions.json", "r") as f:
    constitutions = json.load(f)

nlp = spacy.load("en_core_web_sm")
# Disable components you don't need yet to speed things up
nlp.disable_pipes("ner")

rows = []

for doc_id, text in constitutions.items():
    doc = nlp(text)
    for i, token in enumerate(doc):
        rows.append({
            "doc_id":      doc_id,
            "country":     doc_id.rsplit("_", 1)[0],
            "year":        int(doc_id.rsplit("_", 1)[1]),
            "token_pos":   i,
            "token":       token.text,
            "lower":       token.lower_,
            "lemma":       token.lemma_,
            "pos":         token.pos_,        # coarse POS (NOUN, VERB...)
            "is_stop":     token.is_stop,
            "is_punct":    token.is_punct,
            "is_alpha":    token.is_alpha,
        })

tokens_df = pd.DataFrame(rows)
tokens_df.to_csv("tokens_table.csv", index=False)
print(tokens_df.shape)
print(tokens_df.head())

(5010727, 11)
             doc_id      country  year  token_pos        token        lower  \
0  Afghanistan_2004  Afghanistan  2004          0  Afghanistan  afghanistan   
1  Afghanistan_2004  Afghanistan  2004          1         2004         2004   
2  Afghanistan_2004  Afghanistan  2004          2         \n\n         \n\n   
3  Afghanistan_2004  Afghanistan  2004          3     Preamble     preamble   
4  Afghanistan_2004  Afghanistan  2004          4           \n           \n   

         lemma    pos  is_stop  is_punct  is_alpha  
0  Afghanistan  PROPN    False     False      True  
1         2004    NUM    False     False     False  
2         \n\n  SPACE    False     False     False  
3     Preamble  PROPN    False     False      True  
4           \n  SPACE    False     False     False  


## Generate LIB Table

In [22]:
constitution_data = []
for key in const.keys():
    country = key
    constitution_data.append(country)

In [23]:
import json
import pandas as pd
import nltk
import spacy

# optional if using spaCy later
# nlp = spacy.load("en_core_web_sm")

def tokenize_source(country_name):
    
    # get constitution text for a country
    text = const[country_name]

    # convert full text into lines
    text_lines = text.splitlines()

    # create dataframe
    LINES = pd.DataFrame({
        "country": country_name,
        "line_str": text_lines
    })

    # clean whitespace
    LINES["line_str"] = LINES["line_str"].str.strip()

    # remove empty lines
    LINES = LINES[LINES["line_str"] != ""].copy()

    # ---------- Paragraphs ----------
    # join back together then split on blank lines
    full_text = "\n".join(LINES["line_str"])

    paragraphs = [
        p.strip() for p in full_text.split("\n\n")
        if p.strip()
    ]

    PARAS = pd.DataFrame({
        "country": country_name,
        "para_num": range(len(paragraphs)),
        "para_str": paragraphs
    })

    # ---------- Sentences ----------
    sent_rows = []

    for _, row in PARAS.iterrows():
        sents = nltk.sent_tokenize(row["para_str"])

        for i, sent in enumerate(sents):
            sent_rows.append({
                "country": country_name,
                "para_num": row["para_num"],
                "sent_num": i,
                "sent_str": sent
            })

    SENTS = pd.DataFrame(sent_rows)

    # ---------- Tokens ----------
    token_rows = []

    for _, row in SENTS.iterrows():

        tokens = nltk.word_tokenize(row["sent_str"])
        tagged = nltk.pos_tag(tokens)

        for token_num, (token, pos) in enumerate(tagged):

            term = (
                token.lower()
                .strip()
            )

            # remove punctuation-only tokens
            term = "".join(ch for ch in term if ch.isalnum())

            if term == "":
                continue

            token_rows.append({
                "country": row["country"],
                "para_num": row["para_num"],
                "sent_num": row["sent_num"],
                "token_num": token_num,
                "token_str": token,
                "term_str": term,
                "pos": pos,
                "pos_group": pos[:2]
            })

    TOKENS = pd.DataFrame(token_rows)

    return TOKENS

In [24]:
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")

[nltk_data] Downloading package punkt to /Users/arocha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/arocha/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [25]:
usa_tokens = tokenize_source("United_States_of_America_1992")

In [26]:
usa_tokens

,country,para_num,sent_num,token_num,token_str,term_str,pos,pos_group
0,United_States_of_America_1992,0,0,0,United,united,NNP,NN
1,United_States_of_America_1992,0,0,1,States,states,NNPS,NN
2,United_States_of_America_1992,0,0,2,of,of,IN,IN
3,United_States_of_America_1992,0,0,3,America,america,NNP,NN
4,United_States_of_America_1992,0,0,4,1789,1789,CD,CD
...,...,...,...,...,...,...,...,...
7710,United_States_of_America_1992,0,174,13,appropriate,appropriate,JJ,JJ
7711,United_States_of_America_1992,0,174,14,legislation,legislation,NN,NN
7712,United_States_of_America_1992,0,175,0,Amendment,amendment,NNP,NN
7713,United_States_of_America_1992,0,175,1,XXVII,xxvii,NNP,NN


In [28]:
constitution_data

['Afghanistan_2004',
 'Albania_2008',
 'Algeria_2008',
 'Andorra_1993',
 'Angola_2010',
 'Antigua_and_Barbuda_1981',
 'Argentina_1994',
 'Armenia_2005',
 'Australia_1985',
 'Austria_2009',
 'Azerbaijan_2009',
 'Bahamas_2002',
 'Bahrain_2002',
 'Bangladesh_2011',
 'Barbados_2007',
 'Belarus_2004',
 'Belgium_2012',
 'Belize_2001',
 'Benin_1990',
 'Bhutan_2008',
 'Bolivia_2009',
 'Bosnia_Herzegovina_2009',
 'Botswana_2002',
 'Brazil_2014',
 'Brunei_1984',
 'Bulgaria_2007',
 'Burkina_Faso_2012',
 'Burundi_2005',
 'Cambodia_1999',
 'Cameroon_2008',
 'Canada_2011',
 'Cape_Verde_1992',
 'Central_African_Republic_2010',
 'Chad_2005',
 'Chile_2012',
 'China_2004',
 'Colombia_2005',
 'Comoros_2009',
 'Congo_2001',
 'Costa_Rica_2011',
 'Cote_DIvoire_2009',
 'Croatia_2001',
 'Cuba_2002',
 'Cyprus_2013',
 'Czech_Republic_2002',
 'Democratic_Republic_of_the_Congo_2011',
 'Denmark_1953',
 'Djibouti_2010',
 'Dominica_1984',
 'Dominican_Republic_2010',
 'East_Timor_2002',
 'Ecuador_2008',
 'Egypt_2014'

In [31]:
import os

os.mkdir('data')
for country in constitution_data:
    print("tokenizing: ", country)
    df = tokenize_source(country)
    df.to_csv(f"data/{country}_tokens.csv")

tokenizing:  Afghanistan_2004
tokenizing:  Albania_2008
tokenizing:  Algeria_2008
tokenizing:  Andorra_1993
tokenizing:  Angola_2010
tokenizing:  Antigua_and_Barbuda_1981
tokenizing:  Argentina_1994
tokenizing:  Armenia_2005
tokenizing:  Australia_1985
tokenizing:  Austria_2009
tokenizing:  Azerbaijan_2009
tokenizing:  Bahamas_2002
tokenizing:  Bahrain_2002
tokenizing:  Bangladesh_2011
tokenizing:  Barbados_2007
tokenizing:  Belarus_2004
tokenizing:  Belgium_2012
tokenizing:  Belize_2001
tokenizing:  Benin_1990
tokenizing:  Bhutan_2008
tokenizing:  Bolivia_2009
tokenizing:  Bosnia_Herzegovina_2009
tokenizing:  Botswana_2002
tokenizing:  Brazil_2014
tokenizing:  Brunei_1984
tokenizing:  Bulgaria_2007
tokenizing:  Burkina_Faso_2012
tokenizing:  Burundi_2005
tokenizing:  Cambodia_1999
tokenizing:  Cameroon_2008
tokenizing:  Canada_2011
tokenizing:  Cape_Verde_1992
tokenizing:  Central_African_Republic_2010
tokenizing:  Chad_2005
tokenizing:  Chile_2012
tokenizing:  China_2004
tokenizing: 